# Phase 04I: Teacher B → S2 Full Pseudo-Caption Corpus

Mục tiêu của notebook:

- sinh pseudo-caption xác định bằng **Teacher B BLIP Base seed 42** cho đúng
  **113.287 ảnh Train+Restval**;
- giải mã caption Teacher thành văn bản và mã hóa lại bằng tokenizer word-level
  của Student S2;
- ghi dữ liệu theo shard nguyên tử, có SHA-256 và hỗ trợ resume giữa nhiều
  Kaggle sessions;
- chỉ cấp quyền sang Phase 04J khi corpus đủ, không trùng ảnh, không caption rỗng
  và Held-out Test vẫn `SEALED`.

**Kaggle inputs bắt buộc**

1. Phase 04H output đã publish.
2. Teacher B seed 42 full-run output có checkpoint `best`.
3. Phase 1 formal output.
4. MS COCO 2014 image dataset.
5. Offline model `Salesforce/blip-image-captioning-base` đúng revision.
6. Từ phiên thứ hai: attach **duy nhất output Phase 04I mới nhất** để resume.

**Runtime:** GPU T4 hoặc mạnh hơn; Internet Off. Chạy tuần tự Cell 1 → Cell 8.
Notebook có thể kết thúc ở trạng thái `PARTIAL`; hãy Save Version, attach output
vừa lưu vào phiên tiếp theo và chạy lại từ Cell 1.


## Cell 1 — Environment, authorization và immutable inputs

**Công việc:** kiểm tra GPU, Phase 04H authorization, Teacher checkpoint, Phase 1
contract và offline BLIP; chưa đọc ảnh hoặc Test.

**Đầu ra dự kiến:** `Status: PASSED`, `Progress: 1 / 8 cells passed`.


In [1]:
import csv
import hashlib
import json
import math
import os
import random
import re
import shutil
import time
import unicodedata
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch


CELL_STARTED = time.perf_counter()
SEED = 42
INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path(
    "/kaggle/working//"
    "phase4_teacher_b_s2_pseudo_corpus"
)
AUDIT_DIR = OUTPUT_ROOT / "audits"
CONFIG_DIR = OUTPUT_ROOT / "configs"
DATA_DIR = OUTPUT_ROOT / "data"
SHARD_DIR = DATA_DIR / "shards"
MANIFEST_DIR = OUTPUT_ROOT / "manifests"
for directory in [
    OUTPUT_ROOT, AUDIT_DIR, CONFIG_DIR, DATA_DIR,
    SHARD_DIR, MANIFEST_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

PROGRESS_PATH = (
    OUTPUT_ROOT /
    "PHASE4_TEACHER_B_S2_PSEUDO_CORPUS_PROGRESS.json"
)
SHARDS_MANIFEST_PATH = (
    MANIFEST_DIR / "phase4i_shards_manifest.json"
)


def now_utc():
    return datetime.now(timezone.utc).isoformat()


def read_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)


def write_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    os.replace(temporary, path)


def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def exactly_one(name):
    matches = sorted(INPUT_ROOT.rglob(name))
    assert len(matches) == 1, (name, matches)
    return matches[0].resolve()


def resolve_inside(root, relative_path):
    root = Path(root).resolve()
    target = (root / relative_path).resolve()
    assert target.is_relative_to(root), (root, target)
    return target


def set_deterministic(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.use_deterministic_algorithms(True, warn_only=False)
    torch.backends.cuda.enable_flash_sdp(False)
    torch.backends.cuda.enable_mem_efficient_sdp(False)
    torch.backends.cuda.enable_math_sdp(True)


if not torch.cuda.is_available():
    raise RuntimeError(
        "Phase 04I bắt buộc GPU. Chọn Kaggle GPU T4 rồi chạy lại."
    )
set_deterministic(SEED)
device = torch.device("cuda")

PHASE4H_COMPLETE_PATH = exactly_one(
    "PHASE4_TEACHER_B_S2_SEQUENCE_KD_PREFLIGHT_COMPLETE.json"
)
AUTH_PATH = exactly_one(
    "PHASE4_TEACHER_B_S2_SEQUENCE_KD_PSEUDO_CORPUS_AUTHORIZED.json"
)
TEACHER_MARKER_PATH = exactly_one(
    "TEACHER_B_SEED42_FULL_RUN_COMPLETE.json"
)
PHASE1_READY_PATH = exactly_one(
    "PHASE1_FORMAL_DELIVERABLES_READY.json"
)
phase4h = read_json(PHASE4H_COMPLETE_PATH)
authorization = read_json(AUTH_PATH)
teacher_marker = read_json(TEACHER_MARKER_PATH)
phase1_ready = read_json(PHASE1_READY_PATH)

assert phase4h["status"] == "complete"
assert phase4h["phase04i_authorized"] is True
assert phase4h["manifest_verification"] == "all_passed"
assert authorization["status"] == "authorized"
assert authorization["next_phase"] == (
    "Phase 04I full pseudo-caption corpus generation"
)
assert int(authorization["required_images"]) == 113287
assert authorization["resume_and_sharding_required"] is True
assert teacher_marker["status"] == "complete"
assert teacher_marker["teacher_id"] == "B"
assert int(teacher_marker["seed"]) == 42
assert teacher_marker["qualified_for_kd"] is True
assert phase1_ready["status"] == "ready"
assert phase4h["test_set_status"] == "sealed"
assert phase4h["test_set_used"] is False
assert authorization["test_set_status"] == "sealed"
assert authorization["test_set_used"] is False
assert teacher_marker["test_set_status"] == "sealed"
assert teacher_marker["test_set_used"] is False
assert phase1_ready["test_set_status"] == "sealed"
assert phase1_ready["test_used"] is False

phase4h_root = PHASE4H_COMPLETE_PATH.parent
expected_auth = phase4h["next_authorization"]
assert resolve_inside(
    phase4h_root, expected_auth["relative_path"]
) == AUTH_PATH
assert sha256_file(AUTH_PATH) == expected_auth["sha256"]

TEACHER_ROOT = TEACHER_MARKER_PATH.parent
TEACHER_CHECKPOINT = resolve_inside(
    TEACHER_ROOT,
    teacher_marker["best_checkpoint"]["relative_path"],
)
assert sha256_file(TEACHER_CHECKPOINT) == (
    teacher_marker["best_checkpoint"]["sha256"]
)

model_candidates = []
for manifest in INPUT_ROOT.rglob("MODEL_SOURCE_MANIFEST.json"):
    payload = read_json(manifest)
    if (
        payload.get("repo_id")
        == "Salesforce/blip-image-captioning-base"
        and payload.get("resolved_commit")
        == "82a37760796d32b1411fe092ab5d4e227313294b"
    ):
        model_candidates.append(manifest.parent.resolve())
assert len(model_candidates) == 1, model_candidates
BLIP_SOURCE_DIR = model_candidates[0]
source_manifest = read_json(
    BLIP_SOURCE_DIR / "MODEL_SOURCE_MANIFEST.json"
)
for item in source_manifest["files"]:
    path = BLIP_SOURCE_DIR / item["relative_path"]
    assert path.is_file()
    assert path.stat().st_size == int(item["size_bytes"])
    assert sha256_file(path) == item["sha256"]

previous_progress_candidates = sorted(
    INPUT_ROOT.rglob(
        "PHASE4_TEACHER_B_S2_PSEUDO_CORPUS_PROGRESS.json"
    )
)
assert len(previous_progress_candidates) <= 1, (
    "Chỉ attach output Phase 04I mới nhất.",
    previous_progress_candidates,
)
PRIOR_PROGRESS_PATH = (
    previous_progress_candidates[0].resolve()
    if previous_progress_candidates else None
)

write_json(
    AUDIT_DIR / "phase4i_cell01_inputs.json",
    {
        "status": "passed",
        "created_at_utc": now_utc(),
        "gpu": torch.cuda.get_device_name(0),
        "phase4h": "verified",
        "teacher_checkpoint_sha256": sha256_file(
            TEACHER_CHECKPOINT
        ),
        "offline_blip": "verified",
        "prior_progress_found": PRIOR_PROGRESS_PATH is not None,
        "dataset_read": False,
        "model_loaded": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED,
    },
)
print("=" * 72)
print("PHASE 04I CELL 1 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("GPU                       :", torch.cuda.get_device_name(0))
print("Phase 04H                 : VERIFIED")
print("Teacher checkpoint        : VERIFIED")
print("Offline BLIP              : VERIFIED")
print("Prior Phase 04I progress  :", PRIOR_PROGRESS_PATH is not None)
print("Dataset/model read        : False / False")
print("Test Set status           : SEALED")
print("Progress                  : 1 / 8 cells passed")
print("Next                      : build deterministic image index")


PHASE 04I CELL 1 STATUS
Status                    : PASSED
GPU                       : Tesla T4
Phase 04H                 : VERIFIED
Teacher checkpoint        : VERIFIED
Offline BLIP              : VERIFIED
Prior Phase 04I progress  : False
Dataset/model read        : False / False
Test Set status           : SEALED
Progress                  : 1 / 8 cells passed
Next                      : build deterministic image index


## Cell 2 — Deterministic Train+Restval image index

**Công việc:** đọc metadata của Phase 1, lập đúng 113.287 đường dẫn ảnh Train+Restval,
sắp xếp theo `image_id`; không mở caption hoặc Test Parquet.

**Đầu ra dự kiến:** `Images: 113287`, `Test overlap: 0`.


In [2]:
CELL_STARTED = time.perf_counter()
import pyarrow.parquet as pq


IMAGE_MANIFEST = exactly_one("mscoco_image_manifest.parquet")
VOCAB_PATH = exactly_one("word_vocabulary_minfreq5.json")
vocabulary = read_json(VOCAB_PATH)
id_to_token = vocabulary["id_to_token"]
token_to_id = {
    token: index for index, token in enumerate(id_to_token)
}
assert len(id_to_token) == 10080
tokenizer_pattern = re.compile(vocabulary["tokenizer"]["pattern"])


def student_encode(text):
    normalized = unicodedata.normalize("NFKC", text).lower()
    normalized = normalized.replace("’", "'").replace("‘", "'")
    tokens = tokenizer_pattern.findall(normalized)
    truncated = len(tokens) > 30
    tokens = tokens[:30]
    ids = (
        [1]
        + [token_to_id.get(token, 3) for token in tokens]
        + [2]
    )
    return tokens, ids, truncated


def find_leaf(name, prefix):
    matches = [
        path.resolve()
        for path in INPUT_ROOT.rglob(name)
        if (
            path.is_dir()
            and next(path.glob(f"{prefix}*.jpg"), None)
        )
    ]
    matches = sorted(set(matches))
    assert len(matches) == 1, matches
    return matches[0]


TRAIN_LEAF = find_leaf("train2014", "COCO_train2014_")
VAL_LEAF = find_leaf("val2014", "COCO_val2014_")
manifest_schema = pq.read_schema(IMAGE_MANIFEST)
manifest_columns = set(manifest_schema.names)
required = {
    "image_id", "experiment_split",
    "source_partition", "image_path",
}
assert required.issubset(manifest_columns), manifest_schema

train_rows = pq.read_table(
    IMAGE_MANIFEST,
    columns=[
        "image_id", "source_partition", "image_path",
    ],
    filters=[("experiment_split", "=", "train")],
).to_pylist()
held_out_ids = set(
    int(value)
    for value in pq.read_table(
        IMAGE_MANIFEST,
        columns=["image_id"],
        filters=[
            ("experiment_split", "=", "held_out_test")
        ],
    ).column("image_id").to_pylist()
)
IMAGE_RECORDS = []
for row in train_rows:
    leaf = (
        TRAIN_LEAF
        if row["source_partition"] == "train2014"
        else VAL_LEAF
    )
    image_id = int(row["image_id"])
    IMAGE_RECORDS.append(
        {
            "image_id": image_id,
            "source_partition": row["source_partition"],
            "image_path": (
                leaf / Path(row["image_path"]).name
            ),
        }
    )
IMAGE_RECORDS.sort(key=lambda item: item["image_id"])
IMAGE_IDS = [row["image_id"] for row in IMAGE_RECORDS]
assert len(IMAGE_RECORDS) == 113287
assert len(set(IMAGE_IDS)) == 113287
assert not (set(IMAGE_IDS) & held_out_ids)
for row in (
    IMAGE_RECORDS[:8]
    + IMAGE_RECORDS[-8:]
    + IMAGE_RECORDS[::5000]
):
    assert row["image_path"].is_file(), row["image_path"]

write_json(
    CONFIG_DIR / "phase4i_image_index_contract.json",
    {
        "contract_version": "phase4i_image_index_v1",
        "created_at_utc": now_utc(),
        "status": "locked",
        "ordering": "ascending image_id",
        "images": len(IMAGE_RECORDS),
        "split": "Train+Restval",
        "caption_payload_read": False,
        "held_out_test_overlap": 0,
        "test_set_status": "sealed",
        "test_set_used": False,
    },
)
write_json(
    AUDIT_DIR / "phase4i_cell02_image_index.json",
    {
        "status": "passed",
        "created_at_utc": now_utc(),
        "images": len(IMAGE_RECORDS),
        "unique_images": len(set(IMAGE_IDS)),
        "first_image_id": IMAGE_IDS[0],
        "last_image_id": IMAGE_IDS[-1],
        "held_out_test_overlap": 0,
        "caption_payload_read": False,
        "test_parquet_opened": False,
        "test_images_read": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED,
    },
)
print("=" * 72)
print("PHASE 04I CELL 2 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Images                    :", len(IMAGE_RECORDS))
print("Unique images             :", len(set(IMAGE_IDS)))
print("Ordering                  : ascending image_id")
print("Test overlap              : 0")
print("Caption/Test payload read : False / False")
print("Progress                  : 2 / 8 cells passed")
print("Next                      : load Teacher and benchmark")


PHASE 04I CELL 2 STATUS
Status                    : PASSED
Images                    : 113287
Unique images             : 113287
Ordering                  : ascending image_id
Test overlap              : 0
Caption/Test payload read : False / False
Progress                  : 2 / 8 cells passed
Next                      : load Teacher and benchmark


## Cell 3 — Load frozen Teacher và fixed-sample equivalence

**Công việc:** strict-load Teacher B seed 42, sinh lại bốn ảnh của Phase 04H và
yêu cầu caption/token IDs khớp tuyệt đối. Thời gian đo dùng để đề xuất kích thước
shard khoảng 30 phút.

**Đầu ra dự kiến:** `Repeat exact: 4 / 4`, `Teacher frozen: True`.


In [3]:
CELL_STARTED = time.perf_counter()
from PIL import Image
from transformers import (
    BlipForConditionalGeneration,
    BlipProcessor,
)


PROCESSOR = BlipProcessor.from_pretrained(
    str(BLIP_SOURCE_DIR),
    local_files_only=True,
    use_fast=False,
)
TEACHER = BlipForConditionalGeneration.from_pretrained(
    str(BLIP_SOURCE_DIR),
    local_files_only=True,
)
teacher_payload = torch.load(
    TEACHER_CHECKPOINT,
    map_location="cpu",
    weights_only=False,
)
assert teacher_payload["schema_version"] == (
    "teacher_b_seed42_best_model_v1"
)
assert int(teacher_payload["seed"]) == 42
assert teacher_payload["test_set_status"] == "sealed"
assert teacher_payload["test_set_used"] is False
TEACHER.load_state_dict(
    teacher_payload["model"], strict=True
)
del teacher_payload
TEACHER = TEACHER.to(device).eval()
for parameter in TEACHER.parameters():
    parameter.requires_grad_(False)


def generate_for(rows):
    captions = []
    for start in range(0, len(rows), 4):
        chunk = rows[start:start + 4]
        images = []
        for row in chunk:
            with Image.open(row["image_path"]) as source:
                images.append(source.convert("RGB"))
        pixel_values = PROCESSOR(
            images=images,
            return_tensors="pt",
        ).pixel_values.to(device)
        with torch.inference_mode():
            token_ids = TEACHER.generate(
                pixel_values=pixel_values,
                num_beams=3,
                do_sample=False,
                length_penalty=0.7,
                min_new_tokens=2,
                max_new_tokens=31,
            )
        captions.extend(
            PROCESSOR.batch_decode(
                token_ids, skip_special_tokens=True
            )
        )
    return [caption.strip() for caption in captions]


preflight_sample = read_json(
    resolve_inside(
        phase4h_root,
        "data/phase4h_fixed64_pseudo_captions.json",
    )
)
sample_by_id = {
    int(row["image_id"]): row
    for row in preflight_sample["rows"]
}
record_by_id = {
    row["image_id"]: row for row in IMAGE_RECORDS
}
probe_ids = [
    int(row["image_id"])
    for row in preflight_sample["rows"][:4]
]
probe_rows = [record_by_id[value] for value in probe_ids]
generation_started = time.perf_counter()
probe_captions = generate_for(probe_rows)
probe_seconds = time.perf_counter() - generation_started
for image_id, caption in zip(probe_ids, probe_captions):
    expected = sample_by_id[image_id]
    assert caption == expected["teacher_caption"]
    _, ids, truncated = student_encode(caption)
    assert ids == [
        int(value)
        for value in expected["student_token_ids"]
    ]
    assert truncated == bool(expected["truncated"])

seconds_per_image = probe_seconds / len(probe_rows)
target_shard_seconds = 30 * 60
raw_shard_size = int(
    target_shard_seconds / max(seconds_per_image, 1e-6)
)
proposed = max(64, min(2048, raw_shard_size))
PROPOSED_SHARD_SIZE = max(64, (proposed // 64) * 64)
assert PROPOSED_SHARD_SIZE % 64 == 0

write_json(
    AUDIT_DIR / "phase4i_cell03_teacher_probe.json",
    {
        "status": "passed",
        "created_at_utc": now_utc(),
        "teacher": "BLIP Base seed 42",
        "strict_checkpoint_load": True,
        "repeat_exact_images": 4,
        "probe_seconds": probe_seconds,
        "seconds_per_image": seconds_per_image,
        "proposed_shard_size": PROPOSED_SHARD_SIZE,
        "teacher_frozen": all(
            not p.requires_grad
            for p in TEACHER.parameters()
        ),
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED,
    },
)
print("=" * 72)
print("PHASE 04I CELL 3 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Strict checkpoint load    : PASSED")
print("Repeat exact              : 4 / 4")
print("Teacher frozen            : True")
print("Seconds / image probe     :", round(seconds_per_image, 4))
print("Proposed shard size       :", PROPOSED_SHARD_SIZE)
print("Progress                  : 3 / 8 cells passed")
print("Next                      : restore prior shards")


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForConditionalGeneration LOAD REPORT from: /kaggle/input/models/nguynthnhdan/blip-image-captioning-base-offline/pytorch/82a37760-pytorch/1/salesforce_blip_image_captioning_base_82a37760
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:

PHASE 04I CELL 3 STATUS
Status                    : PASSED
Strict checkpoint load    : PASSED
Repeat exact              : 4 / 4
Teacher frozen            : True
Seconds / image probe     : 0.6056
Proposed shard size       : 2048
Progress                  : 3 / 8 cells passed
Next                      : restore prior shards


## Cell 4 — Restore và kiểm tra các shard đã có

**Công việc:** nếu có output Phase 04I trước, kiểm tra toàn bộ SHA-256 và copy
atomically vào working; xác minh shard không chồng lặp và khớp đúng lát cắt
`image_id`.

**Đầu ra dự kiến:** số ảnh đã resume và vị trí ảnh kế tiếp.


In [4]:
CELL_STARTED = time.perf_counter()


def read_jsonl(path):
    rows = []
    with Path(path).open("r", encoding="utf-8") as handle:
        for line in handle:
            if line.strip():
                rows.append(json.loads(line))
    return rows


def atomic_copy(source, destination):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(
        destination.suffix + ".copying"
    )
    shutil.copy2(source, temporary)
    os.replace(temporary, destination)


def validate_entry(root, entry, expected_shard_size):
    path = resolve_inside(root, entry["relative_path"])
    assert path.is_file()
    assert sha256_file(path) == entry["sha256"]
    rows = read_jsonl(path)
    assert len(rows) == int(entry["row_count"])
    start = int(entry["start_index"])
    end = int(entry["end_index_exclusive"])
    assert end - start == len(rows)
    assert 0 <= start < end <= len(IMAGE_RECORDS)
    assert start % expected_shard_size == 0
    expected_ids = IMAGE_IDS[start:end]
    actual_ids = [int(row["image_id"]) for row in rows]
    assert actual_ids == expected_ids
    assert all(row["teacher_caption"].strip() for row in rows)
    assert all(
        row["student_token_ids"][0] == 1
        and row["student_token_ids"][-1] == 2
        for row in rows
    )
    return path


source_manifests = []
if SHARDS_MANIFEST_PATH.is_file():
    source_manifests.append(
        (OUTPUT_ROOT, read_json(SHARDS_MANIFEST_PATH))
    )
if PRIOR_PROGRESS_PATH is not None:
    prior_root = PRIOR_PROGRESS_PATH.parent
    prior_progress = read_json(PRIOR_PROGRESS_PATH)
    assert prior_progress["experiment_id"] == "P4B_B_S2_SEQKD"
    assert prior_progress["test_set_status"] == "sealed"
    assert prior_progress["test_set_used"] is False
    prior_manifest = read_json(
        resolve_inside(
            prior_root,
            prior_progress["shards_manifest"]["relative_path"],
        )
    )
    assert sha256_file(
        resolve_inside(
            prior_root,
            prior_progress["shards_manifest"]["relative_path"],
        )
    ) == prior_progress["shards_manifest"]["sha256"]
    source_manifests.append((prior_root, prior_manifest))

prior_sizes = {
    int(payload["shard_size"])
    for _, payload in source_manifests
}
assert len(prior_sizes) <= 1, prior_sizes
SHARD_SIZE = (
    next(iter(prior_sizes))
    if prior_sizes else PROPOSED_SHARD_SIZE
)
assert SHARD_SIZE >= 64 and SHARD_SIZE % 64 == 0

entries_by_start = {}
for source_root, payload in source_manifests:
    assert payload["schema_version"] == (
        "phase4i_shards_manifest_v1"
    )
    assert int(payload["total_images"]) == len(IMAGE_RECORDS)
    assert int(payload["shard_size"]) == SHARD_SIZE
    for entry in payload["shards"]:
        source_path = validate_entry(
            source_root, entry, SHARD_SIZE
        )
        destination = SHARD_DIR / source_path.name
        if destination.is_file():
            assert sha256_file(destination) == entry["sha256"]
        else:
            atomic_copy(source_path, destination)
        copied = dict(entry)
        copied["relative_path"] = (
            destination.relative_to(OUTPUT_ROOT).as_posix()
        )
        start = int(copied["start_index"])
        if start in entries_by_start:
            assert (
                entries_by_start[start]["sha256"]
                == copied["sha256"]
            )
        entries_by_start[start] = copied

SHARD_ENTRIES = [
    entries_by_start[key]
    for key in sorted(entries_by_start)
]
expected_start = 0
for entry in SHARD_ENTRIES:
    assert int(entry["start_index"]) == expected_start
    expected_start = int(entry["end_index_exclusive"])
COMPLETED_IMAGES = expected_start
assert 0 <= COMPLETED_IMAGES <= len(IMAGE_RECORDS)

write_json(
    AUDIT_DIR / "phase4i_cell04_resume.json",
    {
        "status": "passed",
        "created_at_utc": now_utc(),
        "prior_progress_found": PRIOR_PROGRESS_PATH is not None,
        "shard_size": SHARD_SIZE,
        "restored_shards": len(SHARD_ENTRIES),
        "restored_images": COMPLETED_IMAGES,
        "next_index": COMPLETED_IMAGES,
        "checksums": "all_passed",
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED,
    },
)
print("=" * 72)
print("PHASE 04I CELL 4 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Shard size                :", SHARD_SIZE)
print("Restored shards           :", len(SHARD_ENTRIES))
print("Restored images           :", COMPLETED_IMAGES)
print("Next index                :", COMPLETED_IMAGES)
print("Checksums                 : ALL PASSED")
print("Progress                  : 4 / 8 cells passed")
print("Next                      : define atomic generation engine")


PHASE 04I CELL 4 STATUS
Status                    : PASSED
Shard size                : 2048
Restored shards           : 0
Restored images           : 0
Next index                : 0
Checksums                 : ALL PASSED
Progress                  : 4 / 8 cells passed
Next                      : define atomic generation engine


## Cell 5 — Atomic generation, manifest và progress engine

**Công việc:** định nghĩa writer JSONL nguyên tử, cập nhật manifest/progress sau
từng shard và tính đầy đủ OOV/truncation.

**Đầu ra dự kiến:** `Engine: READY`; chưa sinh thêm ảnh trong cell này.


In [5]:
CELL_STARTED = time.perf_counter()


def write_jsonl_atomic(path, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("w", encoding="utf-8", newline="\n") as out:
        for row in rows:
            out.write(
                json.dumps(
                    row, ensure_ascii=False, separators=(",", ":")
                )
                + "\n"
            )
        out.flush()
        os.fsync(out.fileno())
    os.replace(temporary, path)


def write_shards_manifest(entries):
    payload = {
        "schema_version": "phase4i_shards_manifest_v1",
        "created_at_utc": now_utc(),
        "experiment_id": "P4B_B_S2_SEQKD",
        "teacher_seed": 42,
        "split": "Train+Restval",
        "total_images": len(IMAGE_RECORDS),
        "shard_size": SHARD_SIZE,
        "completed_images": (
            int(entries[-1]["end_index_exclusive"])
            if entries else 0
        ),
        "shards": entries,
        "test_set_status": "sealed",
        "test_set_used": False,
    }
    write_json(SHARDS_MANIFEST_PATH, payload)
    return payload


def summarize_entries(entries):
    return {
        "rows": sum(int(x["row_count"]) for x in entries),
        "lexical_tokens": sum(
            int(x["lexical_tokens"]) for x in entries
        ),
        "unknown_tokens": sum(
            int(x["unknown_tokens"]) for x in entries
        ),
        "truncated_captions": sum(
            int(x["truncated_captions"]) for x in entries
        ),
        "empty_captions": sum(
            int(x["empty_captions"]) for x in entries
        ),
    }


def write_progress(entries, status):
    manifest = write_shards_manifest(entries)
    summary = summarize_entries(entries)
    completed = int(manifest["completed_images"])
    payload = {
        "marker_version": "phase4i_pseudo_corpus_progress_v1",
        "created_at_utc": now_utc(),
        "status": status,
        "experiment_id": "P4B_B_S2_SEQKD",
        "teacher": "Teacher B — BLIP Base seed 42",
        "student_tokenizer": "Phase 1 word-level minfreq5",
        "split": "Train+Restval",
        "required_images": len(IMAGE_RECORDS),
        "completed_images": completed,
        "remaining_images": len(IMAGE_RECORDS) - completed,
        "shard_size": SHARD_SIZE,
        "completed_shards": len(entries),
        "summary": summary,
        "shards_manifest": {
            "relative_path": (
                SHARDS_MANIFEST_PATH
                .relative_to(OUTPUT_ROOT)
                .as_posix()
            ),
            "sha256": sha256_file(SHARDS_MANIFEST_PATH),
        },
        "resume_required": completed < len(IMAGE_RECORDS),
        "test_set_status": "sealed",
        "test_set_used": False,
    }
    write_json(PROGRESS_PATH, payload)
    return payload


def generate_shard(start, end):
    records = IMAGE_RECORDS[start:end]
    captions = generate_for(records)
    assert len(captions) == len(records)
    output_rows = []
    lexical_tokens = 0
    unknown_tokens = 0
    truncated_captions = 0
    empty_captions = 0
    for record, caption in zip(records, captions):
        caption = caption.strip()
        empty_captions += int(not caption)
        tokens, ids, truncated = student_encode(caption)
        unknown = sum(value == 3 for value in ids[1:-1])
        lexical = len(ids) - 2
        lexical_tokens += lexical
        unknown_tokens += unknown
        truncated_captions += int(truncated)
        output_rows.append(
            {
                "image_id": int(record["image_id"]),
                "teacher_caption": caption,
                "student_token_ids": [int(v) for v in ids],
                "student_lexical_tokens": lexical,
                "unknown_tokens": unknown,
                "truncated": bool(truncated),
                "teacher_seed": 42,
                "split": "Train+Restval",
            }
        )
    assert empty_captions == 0
    assert all(row["student_lexical_tokens"] > 0 for row in output_rows)
    shard_id = start // SHARD_SIZE
    shard_path = SHARD_DIR / (
        f"pseudo_{shard_id:05d}_{start:06d}_{end:06d}.jsonl"
    )
    write_jsonl_atomic(shard_path, output_rows)
    entry = {
        "shard_id": shard_id,
        "start_index": start,
        "end_index_exclusive": end,
        "row_count": len(output_rows),
        "first_image_id": output_rows[0]["image_id"],
        "last_image_id": output_rows[-1]["image_id"],
        "lexical_tokens": lexical_tokens,
        "unknown_tokens": unknown_tokens,
        "truncated_captions": truncated_captions,
        "empty_captions": empty_captions,
        "relative_path": (
            shard_path.relative_to(OUTPUT_ROOT).as_posix()
        ),
        "size_bytes": shard_path.stat().st_size,
        "sha256": sha256_file(shard_path),
    }
    validate_entry(OUTPUT_ROOT, entry, SHARD_SIZE)
    return entry


initial_progress = write_progress(
    SHARD_ENTRIES,
    "complete"
    if COMPLETED_IMAGES == len(IMAGE_RECORDS)
    else "partial",
)
write_json(
    AUDIT_DIR / "phase4i_cell05_engine.json",
    {
        "status": "passed",
        "created_at_utc": now_utc(),
        "atomic_jsonl": True,
        "shard_sha256": True,
        "resume_supported": True,
        "shard_size": SHARD_SIZE,
        "initial_completed_images": COMPLETED_IMAGES,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED,
    },
)
print("=" * 72)
print("PHASE 04I CELL 5 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Engine                    : READY")
print("Atomic shard writes       : True")
print("Resume/checksum           : READY")
print("Initial completed images  :", COMPLETED_IMAGES)
print("Progress                  : 5 / 8 cells passed")
print("Next                      : run budgeted generation")


PHASE 04I CELL 5 STATUS
Status                    : PASSED
Engine                    : READY
Atomic shard writes       : True
Resume/checksum           : READY
Initial completed images  : 0
Progress                  : 5 / 8 cells passed
Next                      : run budgeted generation


## Cell 6 — Budgeted full-corpus generation

**Công việc:** sinh shard liên tiếp trong tối đa 9 giờ; chỉ bắt đầu shard mới nếu
ước lượng còn đủ thời gian. Sau mỗi shard, progress và checksum được ghi ngay.

**Đầu ra dự kiến:** `PARTIAL` hoặc `COMPLETE`. Nếu `PARTIAL`, Save Version rồi
attach output mới nhất vào phiên kế tiếp.


In [6]:
CELL_STARTED = time.perf_counter()
MAX_GENERATION_SECONDS = 9 * 60 * 60
SAFETY_BUFFER_SECONDS = 20 * 60
run_deadline = (
    time.perf_counter()
    + MAX_GENERATION_SECONDS
    - SAFETY_BUFFER_SECONDS
)
entries = list(SHARD_ENTRIES)
next_start = (
    int(entries[-1]["end_index_exclusive"])
    if entries else 0
)
shard_durations = []

print("=" * 72)
print("PHASE 04I FULL PSEUDO-CAPTION GENERATION")
print("=" * 72)
print("Required images           :", len(IMAGE_RECORDS))
print("Already completed         :", next_start)
print("Shard size                :", SHARD_SIZE)
print("Time budget               : 9 hours")
print("Safety buffer             : 20 minutes")

while next_start < len(IMAGE_RECORDS):
    remaining_time = run_deadline - time.perf_counter()
    expected_seconds = (
        float(np.mean(shard_durations))
        if shard_durations
        else max(
            60.0,
            seconds_per_image
            * min(
                SHARD_SIZE,
                len(IMAGE_RECORDS) - next_start,
            ),
        )
    )
    if remaining_time < expected_seconds * 1.15:
        print(
            "Stop before next shard: remaining budget",
            round(remaining_time / 60, 2),
            "minutes.",
        )
        break
    end = min(next_start + SHARD_SIZE, len(IMAGE_RECORDS))
    started = time.perf_counter()
    entry = generate_shard(next_start, end)
    duration = time.perf_counter() - started
    entry["generation_seconds"] = duration
    entries.append(entry)
    shard_durations.append(duration)
    next_start = end
    status = (
        "complete"
        if next_start == len(IMAGE_RECORDS)
        else "partial"
    )
    progress = write_progress(entries, status)
    rate = entry["row_count"] / duration
    remaining = len(IMAGE_RECORDS) - next_start
    eta_hours = (
        remaining / rate / 3600 if rate > 0 else math.inf
    )
    print(
        f"shard={entry['shard_id']:05d} | "
        f"images={next_start}/{len(IMAGE_RECORDS)} | "
        f"seconds={duration:.2f} | "
        f"img/s={rate:.4f} | "
        f"ETA={eta_hours:.2f} h | "
        f"sha={entry['sha256'][:12]}"
    )
    torch.cuda.empty_cache()

final_status = (
    "complete"
    if next_start == len(IMAGE_RECORDS)
    else "partial"
)
GENERATION_PROGRESS = write_progress(entries, final_status)
SHARD_ENTRIES = entries
COMPLETED_IMAGES = next_start
write_json(
    AUDIT_DIR / "phase4i_cell06_generation.json",
    {
        "status": "passed",
        "created_at_utc": now_utc(),
        "corpus_status": final_status,
        "images_before_session": initial_progress[
            "completed_images"
        ],
        "images_after_session": COMPLETED_IMAGES,
        "images_generated_this_session": (
            COMPLETED_IMAGES
            - int(initial_progress["completed_images"])
        ),
        "completed_shards": len(SHARD_ENTRIES),
        "remaining_images": (
            len(IMAGE_RECORDS) - COMPLETED_IMAGES
        ),
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED,
    },
)
print()
print("=" * 72)
print("PHASE 04I CELL 6 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Corpus status             :", final_status.upper())
print("Completed images          :", COMPLETED_IMAGES)
print("Remaining images          :", len(IMAGE_RECORDS) - COMPLETED_IMAGES)
print("Completed shards          :", len(SHARD_ENTRIES))
print("Test Set status           : SEALED")
print("Progress                  : 6 / 8 cells passed")
print(
    "Next                      : finalize corpus"
    if final_status == "complete"
    else "Next                      : package PARTIAL and resume later"
)


PHASE 04I FULL PSEUDO-CAPTION GENERATION
Required images           : 113287
Already completed         : 0
Shard size                : 2048
Time budget               : 9 hours
Safety buffer             : 20 minutes
shard=00000 | images=2048/113287 | seconds=371.57 | img/s=5.5117 | ETA=5.61 h | sha=1bdc8d91b22c
shard=00001 | images=4096/113287 | seconds=378.23 | img/s=5.4147 | ETA=5.60 h | sha=5e96a595dd35
shard=00002 | images=6144/113287 | seconds=379.58 | img/s=5.3954 | ETA=5.52 h | sha=ac7a0ed94ae8
shard=00003 | images=8192/113287 | seconds=378.78 | img/s=5.4069 | ETA=5.40 h | sha=fe3272c86c7b
shard=00004 | images=10240/113287 | seconds=373.31 | img/s=5.4861 | ETA=5.22 h | sha=8df389c65546
shard=00005 | images=12288/113287 | seconds=377.70 | img/s=5.4223 | ETA=5.17 h | sha=267ff628699f
shard=00006 | images=14336/113287 | seconds=378.80 | img/s=5.4066 | ETA=5.08 h | sha=e61bac005539
shard=00007 | images=16384/113287 | seconds=376.59 | img/s=5.4383 | ETA=4.95 h | sha=9200766b4ac0
shard=

## Cell 7 — Final corpus audit và Parquet

**Công việc:** khi đủ 113.287 ảnh, đọc lại toàn bộ shard, kiểm tra uniqueness,
ordering, caption, BOS/EOS, rồi ghi một Parquet chuẩn cho Phase 04J. Nếu corpus
còn partial, cell ghi audit partial và không phát hành Parquet giả.

**Đầu ra dự kiến:** `COMPLETE` hoặc hướng dẫn resume.


In [7]:
CELL_STARTED = time.perf_counter()
FINAL_PARQUET = DATA_DIR / "teacher_b_s2_pseudo_corpus.parquet"
CORPUS_COMPLETE = (
    COMPLETED_IMAGES == len(IMAGE_RECORDS)
)

if CORPUS_COMPLETE:
    import pyarrow as pa
    import pyarrow.parquet as pq

    all_rows = []
    all_ids = []
    for entry in SHARD_ENTRIES:
        path = validate_entry(
            OUTPUT_ROOT, entry, SHARD_SIZE
        )
        rows = read_jsonl(path)
        all_rows.extend(rows)
        all_ids.extend(int(row["image_id"]) for row in rows)
    assert len(all_rows) == 113287
    assert all_ids == IMAGE_IDS
    assert len(set(all_ids)) == 113287
    assert all(row["teacher_caption"].strip() for row in all_rows)
    assert all(
        row["student_token_ids"][0] == 1
        and row["student_token_ids"][-1] == 2
        for row in all_rows
    )
    table = pa.Table.from_pylist(all_rows)
    temporary = FINAL_PARQUET.with_suffix(".parquet.tmp")
    pq.write_table(
        table,
        temporary,
        compression="zstd",
        row_group_size=4096,
    )
    os.replace(temporary, FINAL_PARQUET)
    schema_names = set(pq.read_schema(FINAL_PARQUET).names)
    assert {
        "image_id", "teacher_caption",
        "student_token_ids", "unknown_tokens",
        "truncated", "teacher_seed", "split",
    }.issubset(schema_names)
    assert pq.ParquetFile(FINAL_PARQUET).metadata.num_rows == 113287
    summary = summarize_entries(SHARD_ENTRIES)
    assert summary["rows"] == 113287
    assert summary["empty_captions"] == 0
    final_payload = {
        "status": "passed",
        "corpus_status": "complete",
        "images": 113287,
        "unique_images": 113287,
        "empty_captions": 0,
        "lexical_tokens": summary["lexical_tokens"],
        "unknown_tokens": summary["unknown_tokens"],
        "unknown_rate": (
            summary["unknown_tokens"]
            / summary["lexical_tokens"]
        ),
        "truncated_captions": summary[
            "truncated_captions"
        ],
        "parquet_relative_path": (
            FINAL_PARQUET.relative_to(OUTPUT_ROOT).as_posix()
        ),
        "parquet_sha256": sha256_file(FINAL_PARQUET),
    }
else:
    final_payload = {
        "status": "passed",
        "corpus_status": "partial",
        "images": COMPLETED_IMAGES,
        "remaining_images": (
            len(IMAGE_RECORDS) - COMPLETED_IMAGES
        ),
        "parquet_written": False,
        "resume_required": True,
    }
final_payload.update(
    {
        "created_at_utc": now_utc(),
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED,
    }
)
write_json(
    AUDIT_DIR / "phase4i_cell07_finalize.json",
    final_payload,
)
print("=" * 72)
print("PHASE 04I CELL 7 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Corpus status             :", final_payload["corpus_status"].upper())
print("Images                    :", final_payload["images"])
print("Final Parquet written     :", CORPUS_COMPLETE)
if CORPUS_COMPLETE:
    print("Unknown rate              :", final_payload["unknown_rate"])
    print("Truncated captions        :", final_payload["truncated_captions"])
    print("Parquet SHA-256           :", final_payload["parquet_sha256"])
else:
    print("Resume required           : True")
print("Test Set status           : SEALED")
print("Progress                  : 7 / 8 cells passed")
print("Next                      : package formal output")


PHASE 04I CELL 7 STATUS
Status                    : PASSED
Corpus status             : COMPLETE
Images                    : 113287
Final Parquet written     : True
Unknown rate              : 0.00037705908212025147
Truncated captions        : 1
Parquet SHA-256           : cd6958145624fcaffbb4d7623328cad13c90ae7e16cc8c7729996370a5f3927a
Test Set status           : SEALED
Progress                  : 7 / 8 cells passed
Next                      : package formal output


## Cell 8 — Formal package, marker và Phase 04J authorization

**Công việc:** ghi manifest/checksum cho output hiện tại. Marker Phase 04J chỉ
xuất hiện khi corpus `COMPLETE`; trạng thái partial chỉ tạo marker resume.

**Đầu ra dự kiến:** `READY FOR RESUME` hoặc `PHASE 04J AUTHORIZED`.


In [8]:
CELL_STARTED = time.perf_counter()

if CORPUS_COMPLETE:
    next_auth_path = OUTPUT_ROOT / (
        "PHASE4_TEACHER_B_S2_SEQUENCE_KD_"
        "TRAINING_READINESS_AUTHORIZED.json"
    )
    write_json(
        next_auth_path,
        {
            "marker_version": (
                "phase4_teacher_b_s2_sequence_kd_"
                "training_readiness_authorized_v1"
            ),
            "created_at_utc": now_utc(),
            "status": "authorized",
            "experiment_id": "P4B_B_S2_SEQKD",
            "pseudo_corpus": {
                "relative_path": (
                    FINAL_PARQUET
                    .relative_to(OUTPUT_ROOT)
                    .as_posix()
                ),
                "sha256": sha256_file(FINAL_PARQUET),
                "rows": 113287,
            },
            "next_phase": (
                "Phase 04J S2 Sequence-KD training readiness"
            ),
            "test_set_status": "sealed",
            "test_set_used": False,
        },
    )
    complete_marker = OUTPUT_ROOT / (
        "PHASE4_TEACHER_B_S2_"
        "PSEUDO_CORPUS_COMPLETE.json"
    )
    write_json(
        complete_marker,
        {
            "marker_version": (
                "phase4_teacher_b_s2_pseudo_corpus_complete_v1"
            ),
            "created_at_utc": now_utc(),
            "status": "complete",
            "experiment_id": "P4B_B_S2_SEQKD",
            "teacher_seed": 42,
            "images": 113287,
            "split": "Train+Restval",
            "shards": len(SHARD_ENTRIES),
            "pseudo_corpus": {
                "relative_path": (
                    FINAL_PARQUET
                    .relative_to(OUTPUT_ROOT)
                    .as_posix()
                ),
                "sha256": sha256_file(FINAL_PARQUET),
            },
            "next_authorization": {
                "relative_path": (
                    next_auth_path
                    .relative_to(OUTPUT_ROOT)
                    .as_posix()
                ),
                "sha256": sha256_file(next_auth_path),
            },
            "test_set_status": "sealed",
            "test_set_used": False,
        },
    )
    terminal_marker = complete_marker
else:
    partial_marker = OUTPUT_ROOT / (
        "PHASE4_TEACHER_B_S2_"
        "PSEUDO_CORPUS_PARTIAL.json"
    )
    write_json(
        partial_marker,
        {
            "marker_version": (
                "phase4_teacher_b_s2_pseudo_corpus_partial_v1"
            ),
            "created_at_utc": now_utc(),
            "status": "partial",
            "experiment_id": "P4B_B_S2_SEQKD",
            "completed_images": COMPLETED_IMAGES,
            "required_images": len(IMAGE_RECORDS),
            "remaining_images": (
                len(IMAGE_RECORDS) - COMPLETED_IMAGES
            ),
            "progress": {
                "relative_path": (
                    PROGRESS_PATH
                    .relative_to(OUTPUT_ROOT)
                    .as_posix()
                ),
                "sha256": sha256_file(PROGRESS_PATH),
            },
            "resume_instruction": (
                "Save Version, attach only the latest Phase 04I "
                "output, then rerun Cells 1-8."
            ),
            "test_set_status": "sealed",
            "test_set_used": False,
        },
    )
    terminal_marker = partial_marker

exclude_names = {
    "phase4i_manifest.csv",
    "phase4i_manifest.json",
    "phase4i_SHA256SUMS.txt",
}
artifact_paths = [
    path
    for path in OUTPUT_ROOT.rglob("*")
    if path.is_file() and path.name not in exclude_names
    and not path.name.endswith(".tmp")
    and not path.name.endswith(".copying")
]
artifact_paths.sort(
    key=lambda path: path.relative_to(OUTPUT_ROOT).as_posix()
)
manifest_rows = []
for path in artifact_paths:
    manifest_rows.append(
        {
            "relative_path": (
                path.relative_to(OUTPUT_ROOT).as_posix()
            ),
            "size_bytes": path.stat().st_size,
            "sha256": sha256_file(path),
        }
    )

manifest_csv = MANIFEST_DIR / "phase4i_manifest.csv"
with manifest_csv.open(
    "w", encoding="utf-8", newline=""
) as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=[
            "relative_path", "size_bytes", "sha256"
        ],
    )
    writer.writeheader()
    writer.writerows(manifest_rows)
manifest_json = MANIFEST_DIR / "phase4i_manifest.json"
write_json(
    manifest_json,
    {
        "schema_version": "phase4i_manifest_v1",
        "corpus_status": (
            "complete" if CORPUS_COMPLETE else "partial"
        ),
        "artifact_count": len(manifest_rows),
        "artifacts": manifest_rows,
    },
)
sums_path = MANIFEST_DIR / "phase4i_SHA256SUMS.txt"
temporary = sums_path.with_suffix(".txt.tmp")
with temporary.open(
    "w", encoding="utf-8", newline="\n"
) as handle:
    for row in manifest_rows:
        handle.write(
            f"{row['sha256']}  {row['relative_path']}\n"
        )
    handle.flush()
    os.fsync(handle.fileno())
os.replace(temporary, sums_path)

for row in manifest_rows:
    path = resolve_inside(OUTPUT_ROOT, row["relative_path"])
    assert path.stat().st_size == int(row["size_bytes"])
    assert sha256_file(path) == row["sha256"]

write_json(
    AUDIT_DIR / "phase4i_cell08_package.json",
    {
        "status": "passed",
        "created_at_utc": now_utc(),
        "corpus_status": (
            "complete" if CORPUS_COMPLETE else "partial"
        ),
        "manifest_rows": len(manifest_rows),
        "manifest_verification": "all_passed",
        "terminal_marker": terminal_marker.name,
        "phase04j_authorized": CORPUS_COMPLETE,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED,
    },
)

print("=" * 72)
print("PHASE 04I CELL 8 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print(
    "Corpus status             :",
    "COMPLETE" if CORPUS_COMPLETE else "PARTIAL",
)
print("Completed images          :", COMPLETED_IMAGES)
print("Required images           :", len(IMAGE_RECORDS))
print("Manifest rows             :", len(manifest_rows))
print("Manifest verification     : ALL PASSED")
print("Test Set status           : SEALED")
print("Test Set used             : False")
print("Progress                  : 8 / 8 cells passed")
print("Terminal marker           :", terminal_marker)
if CORPUS_COMPLETE:
    print("Phase 04J                 : AUTHORIZED")
    print(
        "Next                      : build S2 Sequence-KD "
        "training readiness"
    )
else:
    print("Phase 04J                 : NOT AUTHORIZED")
    print(
        "Next                      : Save Version, attach "
        "latest Phase 04I output, rerun"
    )


PHASE 04I CELL 8 STATUS
Status                    : PASSED
Corpus status             : COMPLETE
Completed images          : 113287
Required images           : 113287
Manifest rows             : 69
Manifest verification     : ALL PASSED
Test Set status           : SEALED
Test Set used             : False
Progress                  : 8 / 8 cells passed
Terminal marker           : /kaggle/working//phase4_teacher_b_s2_pseudo_corpus/PHASE4_TEACHER_B_S2_PSEUDO_CORPUS_COMPLETE.json
Phase 04J                 : AUTHORIZED
Next                      : build S2 Sequence-KD training readiness
